In [1]:
import pandas as pd
import pandas.errors
import pandas.core.common
pandas.core.common.SettingWithCopyWarning = pandas.errors.SettingWithCopyWarning
from comorbidipy import comorbidity

pd.set_option('display.expand_frame_repr', False)


df_history = pd.read_csv("../data/inpatient_episode_history.csv")
def age_from_group(s: str) -> int:
    s = str(s)
    if s.endswith("+"):
        return int(s[:-1])
    if s == "0":
        return 0
    lo, hi = s.split("-")
    return (int(lo) + int(hi)) // 2 # "80-84" -> 82

df_history["age_group"] = df_history["age_group"].apply(age_from_group)

cci_df = comorbidity(
    df_history,
    id="id",                      # or "episode_id" if that's the actual name
    code="primary_diagnosis_code",
    score="charlson",
    icd="icd9",
    variant="quan",
    assign0=False,
    age="age_group"
)

print(cci_df.head())
print(cci_df.columns)

# Identify the binary comorbidity columns (usually 17-30 specific codes)
# We exclude 'id', scores, weighted scores (wscore_), and the original age column
cols_to_exclude = ['id', 'comorbidity_score', 'wscore_charlson', 'age_group', 'wscore_age']
comorbidity_cols = [
    c for c in cci_df.columns 
    if c not in cols_to_exclude 
    and not c.startswith('wscore_') #
]
print(comorbidity_cols)
cci_df["num_comorbidities"] = cci_df[comorbidity_cols].sum(axis=1)
print(cci_df[[ "id", "comorbidity_score", "num_comorbidities"]].head())
# extract only relevant columns: id, comorbidity_score, num_comorbidities
cci_df = cci_df[["id", "comorbidity_score", "num_comorbidities"]]
cci_df.to_csv("../data/inpatient_episode_comorbidity.csv", index=False)

       id  age_group  aids  ami  canc  cevd  chf  copd  dementia   hp  ...  mld  pud  pvd  rend  rheumd  diab  diabwc  msld  comorbidity_score  age_adj_comorbidity_score
0  188744         82   0.0  0.0   0.0   0.0  0.0   0.0       0.0  0.0  ...  0.0  0.0  0.0   0.0     0.0   0.0     0.0   0.0                0.0                        4.0
1  188762         57   0.0  0.0   0.0   0.0  0.0   0.0       0.0  0.0  ...  0.0  0.0  0.0   0.0     0.0   0.0     0.0   0.0                0.0                        1.0
2  188763         57   0.0  0.0   0.0   0.0  0.0   0.0       0.0  0.0  ...  0.0  0.0  0.0   0.0     0.0   0.0     0.0   0.0                0.0                        1.0
3  188764         57   0.0  0.0   0.0   0.0  0.0   0.0       0.0  0.0  ...  0.0  0.0  0.0   0.0     0.0   0.0     0.0   0.0                0.0                        1.0
4  188765         57   0.0  0.0   0.0   0.0  0.0   0.0       0.0  0.0  ...  0.0  0.0  0.0   0.0     0.0   0.0     0.0   0.0                0.0        